# 03 - Project: lung segmentation on CT

In this project you train a neural network that finds the lungs in a chest CT slice. At the end you will have
a trained model saved in a file, ready to be used in a web app on the last day.

The project is divided in steps. We work on it for three days:

| Day | Steps |
|---|---|
| 2 | STEP 1 to 7: data, transforms, model, loss |
| 3 | STEP 8 to 11: training, results, saving the model |
| 4 | STEP 12 and 13: test set, using the model on a new image. Then notebook 04 |

The goal for everybody is to reach the end of STEP 13 with a working lung model. If you finish early,
go on to notebook 04 (lung nodules): you do not need to wait for the class.

**How to work**

- Each step starts with a numbered list of things to do.
- The code cell below has one comment for each number. Replace every `...` with your code.
- After each step, "You should see" tells you what the correct result looks like. If your result is different,
  fix it before going on: every step uses the variables of the previous ones.
- Use exactly the variable names written in the instructions.
- Notebooks 01 and 02 contain almost all the code you need. Keep them open.

If you are stuck for more than a few minutes, call a teacher.

# STEP 0: Setup

Nothing to write here: run the cells.

Before you start, check in the right panel of Kaggle: the course dataset is attached (Input), Internet is On,
and the Accelerator is **GPU T4 x2**.

In [ ]:
!pip install -q monai

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

from monai.data import DataLoader
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.networks.nets import UNet
from monai.transforms import (
    Compose,
    EnsureChannelFirst,
    EnsureChannelFirstd,
    Lambdad,
    LoadImage,
    LoadImaged,
    RandAffined,
    Resize,
    Resized,
    ScaleIntensityRange,
    ScaleIntensityRanged,
)
from monai.utils import set_determinism

In [ ]:
# Folder of the course dataset
# On Kaggle the datasets added with "Add Input" are under /kaggle/input, in a subfolder
# (datasets/<owner>/). We look for the course dataset by its name.
DATA_DIR = next(Path("/kaggle/input").rglob("luna16-lung-slices"), Path("/kaggle/input/luna16-lung-slices"))

# Same random numbers at every run, so that everybody gets the same results
set_determinism(seed=42)

# The GPU, if there is one
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

**You should see** `device: cuda`. If you see `cpu`, turn on the GPU (Settings -> Accelerator) and run the
notebook again from the top. Without GPU the training of STEP 9 takes much longer.

# STEP 1: The list of files

We build the list of the patients, as in notebook 02.

1. Create `image_paths`: the sorted list of the `.nii` files in the folder `images`.
2. Create `label_paths`: the same for the folder `labels`.
3. Create the list `dataset_files`: for each patient, a dictionary `{"image": ..., "label": ...}` with the two paths.
4. Print how many patients there are, and the first element of the list.

In [ ]:
# 1. Paths of the images
image_paths = sorted((DATA_DIR / "images").glob("*.nii"))

# 2. Paths of the masks
label_paths = sorted((DATA_DIR / "labels").glob("*.nii"))

# 3. One dictionary per patient
dataset_files = []
for image_path, label_path in zip(image_paths, label_paths):
    dataset_files.append({"image": image_path, "label": label_path})

# 4. Number of patients and first element
print(len(dataset_files))
print(dataset_files[0])

**You should see** the number of patients (888 with the full dataset) and a dictionary with two paths that end
with the same file name, `case_0001.nii`.

# STEP 2: Look at the data

Never train a model on data you have not looked at.

1. Take the patient number 3 of `dataset_files` and save it in `sample`.
2. Load the image with `nibabel` and save the array in `image`.
3. Load the mask and save the array in `mask`.
4. Print the shape of the two arrays.
5. Show the image (lung window: -1000 to 400) and the mask side by side.

In [ ]:
# 1. Patient number 3
sample = dataset_files[3]

# 2. Load the image
image = nib.load(sample["image"]).get_fdata()

# 3. Load the mask
mask = nib.load(sample["label"]).get_fdata()

# 4. Shapes
print(image.shape, mask.shape)

# 5. Image and mask side by side
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.title("Image")
plt.imshow(image.T, cmap="gray", vmin=-1000, vmax=400)
plt.axis("off")

plt.subplot(1, 2, 2)
plt.title("Mask")
plt.imshow(mask.T, cmap="gray")
plt.axis("off")

plt.show()

**You should see** `(512, 512) (512, 512)` and two pictures: the CT slice, and a mask where the two lungs have
two different grey levels (labels 1 and 2).

*Extra, if you are ahead:* change the patient number and look at five or six other patients. Are the lungs
always the same size? Why not?

# STEP 3: Training, validation and test sets

We divide the patients in three groups:

- **training** (70%): the network learns from these;
- **validation** (15%): used during the training to check how the network is doing on images it does not learn from;
- **test** (15%): put aside now and used only once, at the end (STEP 12), to measure the final performance.

`train_test_split` of scikit-learn divides a list in two parts. We use it twice.

1. Divide `dataset_files` in `train_files` (70%) and `other_files` (30%). Use `random_state=42`.
2. Divide `other_files` in two halves: `val_files` and `test_files`. Use `random_state=42`.
3. Print the number of patients in the three groups.

In [ ]:
# 1. 70% training, 30% the rest
train_files, other_files = train_test_split(dataset_files, test_size=0.3, random_state=42)

# 2. Half of the rest for validation, half for test
val_files, test_files = train_test_split(other_files, test_size=0.5, random_state=42)

# 3. Size of the three groups
print(len(train_files), len(val_files), len(test_files))

**You should see** three numbers that add up to the total of STEP 1 (with the full dataset: `621 133 134`).

# STEP 4: Transforms

We write the two pipelines that prepare the data, as in notebook 02.

`val_transforms` is the preprocessing. In this order:

1. load image and label;
2. add the channel to image and label;
3. keep only the lungs in the label (the function `keep_lungs` is already written);
4. rescale the image from the window -1000 / 400 HU to the range 0 / 1, with `clip=True`;
5. resize image and label to 256 x 256, `"bilinear"` for the image and `"nearest"` for the label.

`train_transforms` has the same five steps, plus:

6. a random affine transformation, already written in the cell.

In [ ]:
IMG_SIZE = 256

def keep_lungs(label):
    return ((label == 1) | (label == 2)).float()

In [ ]:
val_transforms = Compose([
    # 1. load image and label
    LoadImaged(keys=["image", "label"]),
    # 2. add the channel
    EnsureChannelFirstd(keys=["image", "label"]),
    # 3. keep only the lungs in the label
    Lambdad(keys="label", func=keep_lungs),
    # 4. rescale the image
    ScaleIntensityRanged(keys="image", a_min=-1000, a_max=400, b_min=0.0, b_max=1.0, clip=True),
    # 5. resize image and label
    Resized(keys=["image", "label"], spatial_size=(IMG_SIZE, IMG_SIZE), mode=("bilinear", "nearest")),
])

train_transforms = Compose([
    # 1. - 5. the same as val_transforms
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Lambdad(keys="label", func=keep_lungs),
    ScaleIntensityRanged(keys="image", a_min=-1000, a_max=400, b_min=0.0, b_max=1.0, clip=True),
    Resized(keys=["image", "label"], spatial_size=(IMG_SIZE, IMG_SIZE), mode=("bilinear", "nearest")),
    # 6. data augmentation
    RandAffined(
        keys=["image", "label"],
        prob=0.5,
        rotate_range=0.2,
        scale_range=0.1,
        translate_range=10,
        mode=("bilinear", "nearest"),
        padding_mode="zeros",
    ),
])

Check your transforms on one patient: run the next cell.

In [ ]:
check = val_transforms(train_files[0])

print("image:", check["image"].shape, "min", check["image"].min().item(), "max", check["image"].max().item())
print("label:", check["label"].shape, "values", np.unique(check["label"]))

**You should see**

```
image: torch.Size([1, 256, 256]) min 0.0 max 1.0
label: torch.Size([1, 256, 256]) values [0. 1.]
```

If the label has other values, look at points 3 and 5 again.

# STEP 5: Dataset and DataLoader

PyTorch reads the data through two objects. (Handbook, section 3.5.)

A **Dataset** answers two questions: how many samples are there, and what is sample number `i`.
The class below does it for our list of patients: when PyTorch asks for sample `i`, it takes the dictionary
number `i` and applies the transforms. Images are read from disk only when they are needed.

In [ ]:
class SegmentationDataset(Dataset):

    def __init__(self, data, transform):
        self.data = data              # the list of dictionaries
        self.transform = transform    # the transforms to apply

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return self.transform(self.data[i])

A **DataLoader** takes the samples from a Dataset and groups them in **batches**: the network sees 16 images
at a time, not one. With `shuffle=True` the order of the patients changes at every epoch.

1. Create `train_ds`, `val_ds` and `test_ds` with `SegmentationDataset`. Training files use `train_transforms`,
   validation and test files use `val_transforms`.
2. Create `train_loader` with `batch_size=16`, `shuffle=True`, `num_workers=2`.
3. Create `val_loader` with `batch_size=16`, `shuffle=False`, `num_workers=2`.
4. Take the first batch of `train_loader` and save it in `batch`.
5. Print the shape of `batch["image"]` and of `batch["label"]`.

In [ ]:
# 1. The three datasets
train_ds = SegmentationDataset(train_files, train_transforms)
val_ds = SegmentationDataset(val_files, val_transforms)
test_ds = SegmentationDataset(test_files, val_transforms)

# 2. Training loader
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)

# 3. Validation loader
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)

# 4. First batch
batch = next(iter(train_loader))

# 5. Shapes
print(batch["image"].shape)
print(batch["label"].shape)

**You should see** `torch.Size([16, 1, 256, 256])` twice: 16 images, 1 channel, 256 x 256 pixels.

The next cell shows the first four images of the batch with their masks. Run it: some images are slightly
rotated or shifted by the augmentation, and the mask must follow the lungs exactly.

In [ ]:
plt.figure(figsize=(16, 4))
for i in range(4):
    plt.subplot(1, 4, i + 1)
    plt.imshow(batch["image"][i, 0].T, cmap="gray")
    plt.imshow(np.ma.masked_equal(batch["label"][i, 0], 0).T, cmap="autumn", alpha=0.3)
    plt.axis("off")
plt.show()

# STEP 6: The model

We use a **U-Net**, the standard network for medical image segmentation. It has an encoder that reduces the
image step by step to understand what is in it, and a decoder that goes back to the original size to say where.
MONAI has it ready: we only choose its size.

1. Create `model` with `UNet`. Fill the three missing arguments:
   - `spatial_dims`: our images are 2D;
   - `in_channels`: the number of channels of our images;
   - `out_channels`: the number of classes, background and lung.
2. Move the model to the GPU with `.to(device)` (already written at the end of the line).
3. Pass the images of `batch` through the model and save the result in `outputs`. The images must be moved
   to the GPU too.
4. Print the shape of `outputs`.

In [ ]:
# 1. and 2. The model
model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=2,
    channels=(16, 32, 64, 128, 256),   # number of filters at each level
    strides=(2, 2, 2, 2),              # the image is halved four times
    num_res_units=2,
).to(device)

# 3. Images through the model
outputs = model(batch["image"].to(device))

# 4. Shape of the output
print(outputs.shape)

**You should see** `torch.Size([16, 2, 256, 256])`.

The output has **two channels**: for each pixel, one score for "background" and one for "lung".
The predicted class of a pixel is the channel with the highest score. The model is not trained yet,
so for now these scores are random.

# STEP 7: Loss, optimizer and metric

Three more objects and we are ready to train. (Handbook, sections 5 and 6: the seminar of tomorrow explains
the optimizer and the learning rate in detail.)

- The **loss** measures how wrong the predictions are. The training makes it smaller. We use the Dice loss:
  0 when prediction and mask overlap perfectly, 1 when they do not overlap at all.
- The **optimizer** changes the weights of the model to reduce the loss. `lr` (learning rate) is the size of each change.
- The **metric** is the number we look at to judge the model: the Dice score, between 0 (no overlap) and 1 (perfect).

1. Create `loss_function` with `DiceLoss(to_onehot_y=True, softmax=True)`.
2. Create `optimizer` with `torch.optim.Adam`, on `model.parameters()`, with `lr=1e-3`.
3. Create `dice_metric` with `DiceMetric()`.
4. Compute the loss of the untrained model on `batch`: `loss_function(outputs, labels)`, where `labels` is
   `batch["label"]` moved to the GPU. Print it with `.item()`.

In [ ]:
# 1. Loss
loss_function = DiceLoss(to_onehot_y=True, softmax=True)

# 2. Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 3. Metric
dice_metric = DiceMetric()

# 4. Loss of the untrained model on one batch
labels = batch["label"].to(device)
loss = loss_function(outputs, labels)
print(loss.item())

**You should see** a number around 0.5 - 0.6: the untrained model is wrong on about half of the overlap.

### Check yourself (end of day 2)

Answer without looking at the notes. If you cannot, read the step again.

1. Why do we keep a test set that we do not use during the training?
2. Why is augmentation in `train_transforms` and not in `val_transforms`?
3. What do the four numbers of `torch.Size([16, 1, 256, 256])` mean?
4. Why does the model have two output channels?

**Answers**

1. To measure the performance on patients that had no influence at all on the model: not on the weights
   (training set) and not on our choices, such as when to stop (validation set).
2. Augmentation creates variety for learning. Validation must measure the same thing at every epoch, on real
   unmodified images.
3. 16 images in the batch, 1 channel, 256 pixels along x, 256 along y.
4. One score per class for each pixel: background and lung.

# STEP 8: One training step

Training repeats the same five operations on one batch after the other:

| | Code | What it does |
|---|---|---|
| a | `optimizer.zero_grad()` | forgets the gradients of the previous step |
| b | `outputs = model(images)` | computes the predictions |
| c | `loss = loss_function(outputs, labels)` | measures the error |
| d | `loss.backward()` | computes how each weight should change (the gradients) |
| e | `optimizer.step()` | changes the weights |

Before writing the full loop, we do one step by hand.

1. Write the five lines a - e in the cell, in this order.
2. Run the cell **ten times** and watch the printed loss.

In [ ]:
images = batch["image"].to(device)
labels = batch["label"].to(device)

# 1. The five lines
optimizer.zero_grad()
outputs = model(images)
loss = loss_function(outputs, labels)
loss.backward()
optimizer.step()

print(loss.item())

**You should see** the loss go down at each run: the model is learning this batch.
If it does not change, one of the five lines is missing or they are in the wrong order.

# STEP 9: The training loop

An **epoch** is one pass over all the training images. At each epoch we:

- **train**: the five lines of STEP 8 for each batch of `train_loader`;
- **validate**: compute the Dice score on `val_loader`, without changing the weights;
- **save** the model if it is the best so far.

The structure of the loop is already written. Complete the parts marked with a number.

1. Training: the five lines of STEP 8.
2. Validation: compute `outputs` with the model.
3. Validation: `preds` is the predicted class of each pixel: `outputs.argmax(dim=1, keepdim=True)`.
4. Saving: if `val_dice` is larger than `best_dice`, update `best_dice` and save the weights with
   `torch.save(model.state_dict(), "lung_segmenter.pth")`.

Then run the cell. With the GPU it takes a few minutes: one line is printed at the end of each epoch.

In [ ]:
MAX_EPOCHS = 20

train_losses = []     # training loss of each epoch
val_dices = []        # validation Dice of each epoch
best_dice = 0

for epoch in range(MAX_EPOCHS):

    # ---------- training ----------
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        # 1. The five lines of STEP 8
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))

    # ---------- validation ----------
    model.eval()
    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            # 2. Predictions of the model
            outputs = model(images)

            # 3. Predicted class of each pixel
            preds = outputs.argmax(dim=1, keepdim=True)

            dice_metric(y_pred=preds, y=labels)

    val_dice = dice_metric.aggregate().item()   # mean Dice over all the validation patients
    dice_metric.reset()
    val_dices.append(val_dice)

    # ---------- saving ----------
    # 4. Save the model if it is the best so far
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), "lung_segmenter.pth")

    print(f"epoch {epoch + 1}/{MAX_EPOCHS}   train loss {train_losses[-1]:.4f}   val dice {val_dice:.4f}   best {best_dice:.4f}")

**You should see** one line per epoch. The training loss goes down and the validation Dice goes up, quickly
in the first epochs and then slowly. After a few epochs the validation Dice is above 0.95; at the end the best one should be between 0.98 and 0.99.

If the Dice stays near 0, or the loss does not go down, stop the cell and check STEP 8 and the points 1 - 3.

Two details of the loop that are worth noticing:

- `model.train()` and `model.eval()` switch the model between its two modes: some layers behave differently
  during training and during evaluation.
- `with torch.no_grad()` tells PyTorch that we are not going to train: it does not compute gradients,
  so validation is faster and uses less memory.

# STEP 10: Learning curves

The two lists filled during the training tell the story of the training.

1. In the first subplot, plot `train_losses`.
2. In the second subplot, plot `val_dices`.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.title("Training loss")
plt.xlabel("epoch")
# 1. Plot the training loss
plt.plot(train_losses)

plt.subplot(1, 2, 2)
plt.title("Validation Dice")
plt.xlabel("epoch")
# 2. Plot the validation Dice
plt.plot(val_dices)

plt.show()

**You should see** a curve that goes down (loss) and one that goes up and then flattens (Dice).

When the Dice curve is flat, more epochs would not help: the model has learned what it can from these data.

# STEP 11: Look at the predictions

A number is not enough: we look at what the model draws.

1. Load the best weights into the model: `model.load_state_dict(torch.load("lung_segmenter.pth"))`.
2. Put the model in evaluation mode.
3. Take the first batch of `val_loader` and save it in `batch`.
4. Inside `torch.no_grad()`, compute `outputs` and then `preds` as in STEP 9.
5. Run the plotting part of the cell, already written.

In [ ]:
# 1. Load the best weights
model.load_state_dict(torch.load("lung_segmenter.pth"))

# 2. Evaluation mode
model.eval()

# 3. First validation batch
batch = next(iter(val_loader))

# 4. Predictions
with torch.no_grad():
    outputs = model(batch["image"].to(device))
    preds = outputs.argmax(dim=1, keepdim=True)

# 5. Image, reference mask and predicted mask for the first three patients of the batch
preds = preds.cpu()
plt.figure(figsize=(12, 12))
for i in range(3):
    plt.subplot(3, 3, 3 * i + 1)
    plt.title("Image")
    plt.imshow(batch["image"][i, 0].T, cmap="gray")
    plt.axis("off")

    plt.subplot(3, 3, 3 * i + 2)
    plt.title("Reference mask")
    plt.imshow(batch["label"][i, 0].T, cmap="gray")
    plt.axis("off")

    plt.subplot(3, 3, 3 * i + 3)
    plt.title("Prediction")
    plt.imshow(preds[i, 0].T, cmap="gray")
    plt.axis("off")
plt.show()

**You should see** three rows. In each row the prediction should be very similar to the reference mask.

### Download your model

The file `lung_segmenter.pth` contains the weights of your trained model. You need it on day 5 for the web app.

1. In the right panel of Kaggle open **Output**: the file is in `/kaggle/working`.
2. Click the three dots next to `lung_segmenter.pth` and choose **Download**.
3. Keep the file in a safe place on your computer.

Do it now: when the Kaggle session ends, the files in `/kaggle/working` are deleted.

### Check yourself (end of day 3)

1. What does each of the five lines of the training step do?
2. Why do we save the model when the **validation** Dice improves, and not when the training loss improves?
3. What would you expect to see in the curves if we trained for 200 epochs?

**Answers**

1. See the table of STEP 8.
2. The training loss is measured on images the model has already seen: it can keep improving just because
   the model memorises them. The validation Dice tells how the model does on new patients, which is what we want.
3. The training loss keeps going down slowly. The validation Dice stays flat and may start to go down:
   the model is overfitting.

# STEP 12: The test set

Until now the test patients were never used. We measure the Dice score of each test patient: this is the number
to report as the performance of the model.

The loader below gives one patient at a time, so each Dice value belongs to one patient.

1. Inside the loop, compute `outputs` and `preds` as in STEP 9.
2. After the loop, `dice_metric.aggregate()` would give the mean. We want the value of each patient instead:
   this is already written, in `test_dices`.
3. Print the mean, the minimum and the maximum of `test_dices`.
4. Plot the histogram of `test_dices`.

In [ ]:
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2)

model.eval()
with torch.no_grad():
    for batch in test_loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        # 1. Predictions
        outputs = model(images)
        preds = outputs.argmax(dim=1, keepdim=True)

        dice_metric(y_pred=preds, y=labels)

# 2. One Dice value per test patient
test_dices = dice_metric.get_buffer().cpu().numpy().flatten()
dice_metric.reset()

# 3. Mean, minimum and maximum
print("mean:", test_dices.mean())
print("min :", test_dices.min())
print("max :", test_dices.max())

# 4. Histogram
plt.hist(test_dices, bins=20)
plt.xlabel("Dice")
plt.ylabel("number of patients")
plt.show()

**You should see** a mean Dice of about 0.97 - 0.98 and a histogram with almost all the patients close to 1 and a
very few much lower.

The mean hides the patients where the model fails. The next cell, already written, shows the three test
patients with the lowest Dice. Run it and look at them.

In [ ]:
worst = np.argsort(test_dices)[:3]

plt.figure(figsize=(12, 12))
for row, i in enumerate(worst):
    sample = test_ds[i]
    with torch.no_grad():
        pred = model(sample["image"][None].to(device)).argmax(dim=1)[0].cpu()

    plt.subplot(3, 3, 3 * row + 1)
    plt.title(f"Dice {test_dices[i]:.3f}")
    plt.imshow(sample["image"][0].T, cmap="gray")
    plt.axis("off")

    plt.subplot(3, 3, 3 * row + 2)
    plt.title("Reference mask")
    plt.imshow(sample["label"][0].T, cmap="gray")
    plt.axis("off")

    plt.subplot(3, 3, 3 * row + 3)
    plt.title("Prediction")
    plt.imshow(pred.T, cmap="gray")
    plt.axis("off")
plt.show()

Look at the three cases and discuss them with your neighbour:

- What do they have in common? (Size of the lungs? Position of the slice?)
- Is the model wrong, or is the reference mask wrong? Remember that the masks were made by an automatic algorithm.

# STEP 13: Use the model on a new image

In the web app the model receives a file it has never seen, with no mask. We write the function that the app
will use: from the path of a CT slice to the predicted mask.

The preprocessing must be **exactly the same** as in the training, otherwise the model receives images different
from the ones it has learned from, and it makes mistakes. Here there is no label, so we use the transforms
without the final `d`, which work on one image and not on a dictionary.

1. Complete `inference_transforms` with the same steps as `val_transforms`, for the image only:
   load, add the channel, rescale the intensities, resize.
2. Complete the function `segment(image_path)`:
   - apply `inference_transforms` to the path;
   - add the batch dimension with `[None]` and move the image to the GPU;
   - compute `outputs` with the model, inside `torch.no_grad()`;
   - `pred` is the predicted class of each pixel.
3. Try the function on the first test patient, with the cell below.

In [ ]:
# 1. The preprocessing, for one image
inference_transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),
    ScaleIntensityRange(a_min=-1000, a_max=400, b_min=0.0, b_max=1.0, clip=True),
    Resize(spatial_size=(IMG_SIZE, IMG_SIZE), mode="bilinear"),
])

# 2. From a file to a mask
def segment(image_path):
    image = inference_transforms(image_path)          # shape (1, 256, 256)
    batch = image[None].to(device)                    # shape (1, 1, 256, 256)
    model.eval()
    with torch.no_grad():
        outputs = model(batch)
    pred = outputs.argmax(dim=1)[0].cpu()             # shape (256, 256)
    return image[0], pred

In [ ]:
# 3. Try it on the first test patient
image, pred = segment(test_files[0]["image"])

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Image")
plt.imshow(image.T, cmap="gray")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.title("Prediction")
plt.imshow(image.T, cmap="gray")
plt.imshow(np.ma.masked_equal(pred, 0).T, cmap="autumn", alpha=0.4)
plt.axis("off")
plt.show()

**You should see** the CT slice and, on the right, the same slice with the predicted lungs coloured.

The project is complete: you have a trained model, you know how good it is, and you have the function to use it.
On day 5 the same transforms and the same model go into the web app.

### Check yourself (end of day 4)

1. Your mean test Dice is about 0.98. What would you say to a doctor who asks "does it work"?
2. What happens if the web app rescales the image with a different window, for example -160 / 240?
3. Which files do you need to bring to day 5?

**Answers**

1. On average the overlap with the reference masks is very high, but: a few patients are clearly worse (show
   them), the reference masks are themselves automatic, and all patients come from one public dataset. It is a
   good student project, not a clinical tool.
2. The model receives images that look different from the training ones and the predictions get worse, without
   any error message. Try it: change the numbers in `inference_transforms`.
3. `lung_segmenter.pth`. The preprocessing is already written in the app.

# If you have finished

Your lung model is done. Lungs are an easy target: almost any reasonable choice gives a good result, so there
is little to gain from more experiments here.

**Go to notebook 04, lung nodules.** It uses the same code on a much harder problem, where your choices change
the result, and it is built around experiments.

If you prefer to stay on the lungs for a while, three short challenges:

- **A threshold instead of a network.** Lung is air: predict "lung" where the image is below -400 HU.
  Compute its Dice on the test set, as in STEP 12. Is the neural network better? Where does the threshold fail?
- **Right and left lung.** Three classes instead of two: change `keep_lungs` so that it keeps the values 1 and 2
  as they are (right and left lung) and sets only the trachea to 0, and use `out_channels=3`. Train again from STEP 6.
- **Find the bugs.** Ask a teacher for the training loop with three mistakes, and find them.

# To go further

- U-Net, the original paper: https://arxiv.org/abs/1505.04597
- Dice loss (V-Net paper): https://arxiv.org/abs/1606.04797
- MONAI tutorial, the same pipeline in 3D on the spleen: https://github.com/Project-MONAI/tutorials/blob/main/3d_segmentation/spleen_segmentation_3d.ipynb
- MONAI losses: https://docs.monai.io/en/stable/losses.html
- MONAI metrics: https://docs.monai.io/en/stable/metrics.html
- A recipe for training neural networks (A. Karpathy): https://karpathy.github.io/2019/04/25/recipe/
- Why one number is not enough to evaluate a segmentation, Metrics Reloaded: https://arxiv.org/abs/2206.01653